# Next-Word Prediction in Philosophy of Religion
Compare count-based models, Baum-Welch HMMs and Elman RNNs on William James's *The Varieties of Religious Experience*.

The repository includes the corpus, all trained models, measured results and the PDF report. **Run cells 1 and 2 to predict immediately. Retraining is optional.** A GPU is unnecessary; the supplied experiment uses deterministic CPU training.


In [ ]:
# 1. Get the complete project and install its two core dependencies.
from pathlib import Path
import os, subprocess
project = Path('/content/philosophy-autocomplete')
if not (project / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1',
        'https://github.com/AviBer27/philosophy-autocomplete.git', str(project)], check=True)
os.chdir(project)
%pip -q install -r requirements.txt


In [ ]:
# 2. Use the supplied validation-selected model without training.
!python predict.py --text "religious experience is" --top-k 5
!python predict.py --text "the meaning of" --top-k 5


In [ ]:
# 3. Compare the saved count-based, HMM and RNN models.
!python predict.py --model output/models/trigram.json --text "a sense of"
!python predict.py --model output/models/hmm_16_s19.json --text "a sense of"
!python predict.py --model output/models/rnn_64_s7.json --text "a sense of"


In [ ]:
# 4. Change the input below. The optional prefix belongs to the next word.
from predict import resolve_model, suggest
from autocomplete.storage import load_model
model, vocabulary, metadata = load_model(resolve_model())
suggest(model, vocabulary, text="the human mind", top_k=5, prefix="")


## Optional: reproduce the experiment
The following cell trains all eleven retained configurations (plus the n-gram validation search), selects using validation data and finally scores the held-out test data. It saves a separate run and preserves the supplied results. Runtime varies by machine. A different vocabulary changes the prediction space; perplexities should be compared within the same experiment.


In [ ]:
!python experiment.py --output output_reproduced


In [ ]:
# Inspect the supplied reference results, or change to output_reproduced/results.json.
import json
results = json.loads(Path('output/results.json').read_text())
print('Selected by validation NLL:', results['selected_model']['name'])
for row in results['models']:
    print(f"{row['name']:14} test PPL={row['test']['perplexity']:.2f} "
          f"top-3={100*row['test']['top3']:.2f}% top-5={100*row['test']['top5']:.2f}%")


In [ ]:
# Run the implementation and evaluation checks.
!python -m unittest discover -s tests -v


## Report
The supplied PDF describes the supplied measured experiment. Download it below. If you retrain, rebuild the report against the new results before using it to describe that run.


In [ ]:
from google.colab import files
files.download('output/pdf/project_report.pdf')


In [ ]:
# Optional report regeneration after the reproduction cell.
%pip -q install -r requirements-report.txt
!python scripts/build_report.py --results output_reproduced/results.json --output output_reproduced/project_report.pdf --repository-url https://github.com/AviBer27/philosophy-autocomplete
